# RAFTcorr on Google Colab

Learned digital image correlation with the RAFTcorr-s1, -s2, -s4 and -s8 models, through the
Python API. Use a GPU runtime (*Runtime > Change runtime type > GPU*).

Conventions: `(u, v)` is the reference -> deformed displacement, `+u` right, `+v` down, in pixels,
in frame-0 (Lagrangian) coordinates; frame `k` of a result is image `k` of the sequence.

In [ ]:
!git clone -q https://github.com/zachtong/RAFTcorr.git
%cd RAFTcorr
!pip install -q -e ".[hdf5]"   # Colab already has PyTorch with CUDA

## Model weights

Put `RAFTcorr-s1.pth` ... `RAFTcorr-s8.pth` into `RAFTcorr/models/` (upload them below, or copy
them from Google Drive). `raftcorr models` shows what is installed.

In [ ]:
from google.colab import files
import shutil, os
os.makedirs("models", exist_ok=True)
for name in files.upload():          # choose the .pth files
    shutil.move(name, os.path.join("models", name))
!raftcorr models
!raftcorr selftest

## Your images

Upload a zip of the image folder (frame 0 = first file in natural order) and, optionally, an ROI
mask image (white = inside).

In [ ]:
import zipfile
uploaded = files.upload()            # images.zip (+ roi.png)
zip_name = next(n for n in uploaded if n.endswith(".zip"))
zipfile.ZipFile(zip_name).extractall("images")
folder = "images"
for root, dirs, fnames in os.walk("images"):   # use the first folder that holds images
    if any(f.lower().endswith((".png", ".tif", ".tiff", ".bmp", ".jpg")) for f in fnames):
        folder = root
        break
roi_file = next((n for n in uploaded if n.lower().endswith(".png") and "roi" in n.lower()), None)
print(folder, roi_file)

## Which model?

A quick s8 survey measures the motion and suggests the finest model that reaches it.

In [ ]:
!raftcorr suggest "$folder" {"--roi " + roi_file if roi_file else ""}

In [ ]:
import raftcorr
from raftcorr.config import RunConfig, TrackingConfig

config = RunConfig(model="s4", tracking=TrackingConfig(mode="accumulative"))   # or "incremental"
series = raftcorr.track(folder, roi=roi_file, config=config, callbacks=raftcorr.ConsoleCallbacks())
strain = raftcorr.compute_strain(series)       # Green-Lagrange, uniform 41 px gauge, plane fit
print(series.info.notes)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

k = series.n_frames - 1
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, (name, data, unit) in zip(ax, [("u", series.u[k], "px"), ("v", series.v[k], "px"),
                                       ("exx", strain.components["exx"][k], "-")]):
    im = a.imshow(data, cmap="turbo")
    a.set_title(f"{name}, frame {k}")
    a.axis("off")
    plt.colorbar(im, ax=a, label=f"{name} ({unit})", fraction=0.046)
plt.tight_layout()

In [ ]:
from raftcorr.export import export_results
export_results("results.h5", series, strain, overwrite=True)      # also .mat (MATLAB v7.3) or .npz
files.download("results.h5")